# Block permutation — 救活晚期窗口的无偏方法

接在 `window_fi_figures.ipynb` 之后。**自给自足**,不依赖上一个 notebook 的 kernel 状态。

## 为什么需要

正式跑的结果里有个死结:

| 方法 | 晚期窗口地板 (top-15 Jaccard) | 是否有偏 |
|---|---|---|
| Gini | 0.875 | **有偏**(偏向被频繁分裂的特征) |
| SHAP | 1.000 | **有偏**(与 Gini 同源,ρ=0.97) |
| Permutation | **0.034** | 无偏 |

**唯一无偏的方法在晚期窗口不可用**,所以现在报告里的 ρ ≈ 0.39 只能当上界。

根因诊断已经很明确:晚期窗口 **42.5%** 的 permutation 重要性恰好为 0(TS 是 13.2%)。
不是"特征不重要",而是**互相掩盖**——打乱 GLY214,和它相关 0.9 的邻居还在,
准确率不掉,重要性归零。

## 对策

把高相关的特征聚成**块**,整块一起打乱(同一行序作用于块内所有列,保留块内结构)。
冗余伙伴同时被破坏,掩盖效应消失。

**这仍然是 permutation ——量的还是 held-out 预测贡献,仍然无偏。**
改变的只是"什么算一个坐标"。

## 两个设计决定

1. **块在两个窗口之间共用。** 分别聚类的话块的定义不同,跨窗口对比会变成苹果比橘子。
   这里用两个窗口 Spearman 距离矩阵的平均聚一次。
2. **所有分析在块层面做**,不广播回特征。广播会让同块特征取值完全相同,
   top-k 里全是并列,Jaccard 又变回噪声。

## 判断成功的标准

跑完看**晚期窗口的块级地板**:

- 明显高于 0.034(比如 > 0.3)→ **成功**,有了在两个窗口都可用的无偏方法,
  可以用它复核 ρ ≈ 0.39,把上界变成真值
- 仍然很低 → 冗余不是唯一原因,晚期窗口用无偏方法确实测不出来,
  报告里 SHAP 的上界说法就是最终结论

In [1]:
import json, os, time, warnings
from itertools import combinations

import h5py
import numpy as np
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from scipy.stats import fisher_exact, spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold

# ===== 必须和 window_fi_contrast 那次正式跑完全一致,否则模型对不上 =====
H5_PATH     = "/mnt/data1/student/trypsin/h5_datasets/CV_database_4864_TS_variedDCDfreq_lmin2p6_lmax4p0.h5"
PDB_PATH    = "/mnt/data1/student/trypsin/h5_datasets/step3_input.pdb"
WATERS_PATH = "/mnt/data1/student/trypsin/h5_datasets/4864_TS_50closest_waters.npy"
OUT_DIR     = "/mnt/data1/student/trypsin/yucheng/07.12/v4"

WINDOWS   = {"TS": (500, 1501), "late": (2000, 2500)}
RF_PARAMS = dict(n_estimators=300, max_features="sqrt", min_samples_leaf=2,
                 class_weight="balanced_subsample", n_jobs=-1)
N_SPLITS, SEEDS = 5, list(range(10))
N_PROTEIN, N_WATERS = 224, 50

# ===== block permutation 专有参数 =====
BLOCK_REPEATS = 10        # 每块打乱几次(和主流程的 PERM_REPEATS 一致)
TOP_K         = 15        # top-k Jaccard 的 k

# 合并阈值:距离 = 1 - |Spearman rho|,所以 0.30 表示 |rho| > 0.70 合并。
# 这个值直接决定块的粗细。下面会打印块数和大小分布,不合适就调:
#   调小(0.20) -> 块更多更细,更接近逐特征 permutation
#   调大(0.40) -> 块更少更粗,掩盖问题解决得更彻底但分辨率下降
BLOCK_DISTANCE = 0.30

# 是否禁止水和蛋白残基混进同一个块。默认 True:
# guide §1 要求把水当作独立的一条 track,混进去之后"水的富集"统计就没法解释了。
SEPARATE_WATERS = True

print("配置就绪。RF_PARAMS / SEEDS / N_SPLITS 必须和主 notebook 一致。")

配置就绪。RF_PARAMS / SEEDS / N_SPLITS 必须和主 notebook 一致。


## Part 0 — 载入数据、重建 split

不重训任何主流程的模型,只重新构造特征矩阵和折。

In [2]:
t0 = time.time()
with h5py.File(H5_PATH, "r") as h:
    cv_data = h["cv_data"][:]; labels_raw = h["labels"][:]; groups = h["replica_ids"][:]
labels = np.array([v.decode(errors="replace") if isinstance(v, (bytes, np.bytes_)) else str(v)
                   for v in labels_raw])
y = np.array([0 if l == "IN" else 1 for l in labels], dtype=np.int64)
Xs = {k: cv_data[:, a:b, :].mean(axis=1).astype(np.float64) for k, (a, b) in WINDOWS.items()}
del cv_data

import mdtraj as md
residues = [r for r in md.load(PDB_PATH).topology.residues if r.is_protein]
waters = np.load(WATERS_PATH, allow_pickle=True)
names = np.concatenate([[f"{r.name}{r.resSeq}" for r in residues],
                        [f"water rank{i}" for i in range(N_WATERS)]])
kind = np.array(["protein"] * N_PROTEIN + ["water"] * N_WATERS)
IS_WAT = kind == "water"

def grouped_splits(y, groups, n_splits, seed):
    sp = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return [(tr, te) for tr, te in sp.split(np.arange(len(y)), y, groups)]

# 参照值:逐特征 permutation 的地板(从主流程的 npz 读,用于最后的对比表)
z = dict(np.load(os.path.join(OUT_DIR, "fi_per_seed.npz"), allow_pickle=True))
per_seed_feat = {w: {m: z[f"{w}__{m}"] for m in ("gini", "perm", "shap") if f"{w}__{m}" in z}
                 for w in WINDOWS}
print(f"n_traj={len(y)}  IN={(y==0).sum()}  OUT={(y==1).sum()}  ({time.time()-t0:.0f}s)")
for w in WINDOWS:
    print(f"  {w:6s} X={Xs[w].shape}  已有方法={sorted(per_seed_feat[w])}")

n_traj=173  IN=99  OUT=74  (3s)
  TS     X=(173, 274)  已有方法=['gini', 'perm', 'shap']
  late   X=(173, 274)  已有方法=['gini', 'perm', 'shap']


## Part 1 — 建共用块

对每个窗口算 274×274 的 Spearman 相关矩阵,取 `1 - |rho|` 作距离,**两个窗口取平均**,
再做 average-linkage 层次聚类,按 `BLOCK_DISTANCE` 切。

用平均距离矩阵是为了让两个窗口共用同一套块 —— 否则跨窗口比的就不是同一批坐标了。

In [3]:
t0 = time.time()
# 每个窗口各自的距离矩阵
Dw = {w: 1.0 - np.abs(spearmanr(Xs[w]).statistic) for w in WINDOWS}

# 取【最小】距离(= 最大 |rho|),不是平均。
# 平均是错的:只在一个窗口里相关的特征永远合并不了 —— 而掩盖问题恰恰就发生在
# 那个窗口。例如 GLY214/CYS215/ALA216 在晚期高度相关、在 TS 不相关,平均后
# 距离 ≈ 0.5 不会合并,晚期的掩盖问题就没解决。
# 取最小意味着"在任一窗口相关就合并"。合并后的块在两个窗口共用,
# 在不相关的那个窗口里整块打乱依然有效,只是不必要而已 —— 代价小,漏掉才是大问题。
D = np.minimum.reduce([Dw[w] for w in WINDOWS])
D = (D + D.T) / 2.0
np.fill_diagonal(D, 0.0)
D = np.clip(D, 0, None)

if SEPARATE_WATERS:
    # 把跨类型的距离设成最大,聚类就不会把水和残基并进同一块
    cross = IS_WAT[:, None] != IS_WAT[None, :]
    D[cross] = D.max() + 1.0
    np.fill_diagonal(D, 0.0)

Z = linkage(squareform(D, checks=False), method="average")
labels_cl = fcluster(Z, t=BLOCK_DISTANCE, criterion="distance")
blocks = [np.where(labels_cl == c)[0] for c in np.unique(labels_cl)]
sizes = np.array([len(b) for b in blocks])

# 诊断:每个窗口单独看会合并多少对,确认取 min 是必要的
thr = 1.0 - BLOCK_DISTANCE
iu = np.triu_indices(len(names), k=1)
for w in WINDOWS:
    n_pairs = int((np.abs(1.0 - Dw[w])[iu] > thr).sum())
    print(f"  |rho| > {thr:.2f} 的特征对: {w:6s} {n_pairs}")
print(f"  取 min 之后合并成的块数 = {len(blocks)}  (原本 274 个特征)   "
      f"[{time.time()-t0:.0f}s]")
print(f"块大小: 中位数 {np.median(sizes):.0f}, 最大 {sizes.max()}, "
      f"单特征块 {int((sizes == 1).sum())} 个")
print(f"预计加速: {274 / len(blocks):.1f}x  (permutation 开销 ∝ 块数)\n")

print(f"最大的 8 个块:")
for b in sorted(blocks, key=len, reverse=True)[:8]:
    n_w = int(IS_WAT[b].sum())
    tag = f"  [{n_w} 水]" if n_w else ""
    print(f"  size={len(b):2d}{tag}: " + ", ".join(names[i] for i in b[:8])
          + (" ..." if len(b) > 8 else ""))

block_kind = np.array(["water" if IS_WAT[b].all() else
                       ("protein" if not IS_WAT[b].any() else "mixed") for b in blocks])
print(f"\n块的类型: protein {int((block_kind=='protein').sum())}, "
      f"water {int((block_kind=='water').sum())}, mixed {int((block_kind=='mixed').sum())}")
if (block_kind == "mixed").sum() and SEPARATE_WATERS:
    warnings.warn("SEPARATE_WATERS=True 但仍出现 mixed 块 —— 检查掩码逻辑")

  |rho| > 0.70 的特征对: TS     4642
  |rho| > 0.70 的特征对: late   5216
  取 min 之后合并成的块数 = 73  (原本 274 个特征)   [0s]
块大小: 中位数 1, 最大 74, 单特征块 61 个
预计加速: 3.8x  (permutation 开销 ∝ 块数)

最大的 8 个块:
  size=74: VAL20, GLY21, GLY22, SER122, LEU123, PRO124, THR125, SER126 ...
  size=37: ALA56, ALA57, HIS58, CYS59, TYR60, LYS87, SER88, ILE89 ...
  size=33: GLY26, ALA27, ASN28, GLN33, VAL34, SER35, HIS41, GLY44 ...
  size=18: LEU36, ASN37, SER38, GLY39, PHE42, CYS43, LYS61, SER62 ...
  size=15: GLY45, SER50, GLN51, TRP52, VAL53, SER55, LEU108, SER110 ...
  size=12: PRO31, TYR32, SER46, LEU47, ILE48, ASN49, VAL54, LEU114 ...
  size= 8: ILE19, THR24, CYS25, THR29, VAL30, SER137, LYS154, CYS155
  size= 8: MET178, TYR230, SER232, ILE234, LYS235, GLN236, ILE238, ALA239

块的类型: protein 24, water 49, mixed 0


## Part 2 — block permutation 主循环

核心操作:

```python
order = rng.permutation(n_test)
Xp[:, cols] = X_test[order][:, cols]   # 同一行序作用于块内所有列
```

**同一个行序**很关键 —— 它保留了块内特征之间的相关结构,只切断整块与标签的关系。
如果每列各自独立打乱,破坏的就不只是块与标签的联系,还有块内部的结构,
测出来的东西就不是"这组坐标携带多少信息"了。

模型与主流程完全一致(`random_state = 10_000*seed + fold`),所以拟合出的森林
逐棵树相同,块级重要性和逐特征 permutation 可以直接对比。

预计 15–20 分钟。跑完立刻存盘。

In [4]:
def block_permutation_importance(model, X_te, y_te, blocks, n_repeats, rng):
    """整块打乱,返回每块的 balanced-accuracy 下降量(长度 = 块数)。"""
    base = balanced_accuracy_score(y_te, model.predict(X_te))
    imp = np.zeros(len(blocks))
    n = len(X_te)
    for bi, cols in enumerate(blocks):
        drops = np.empty(n_repeats)
        for r in range(n_repeats):
            Xp = X_te.copy()
            order = rng.permutation(n)
            Xp[:, cols] = X_te[np.ix_(order, cols)]      # 同一行序 -> 保留块内结构
            drops[r] = base - balanced_accuracy_score(y_te, model.predict(Xp))
        imp[bi] = drops.mean()
    return imp


t0 = time.time()
per_seed_blk = {}
oof_blk = {}
for wkey in WINDOWS:
    rows, oofs = [], []
    for seed in SEEDS:
        rng = np.random.default_rng(30_000 + 100 * seed)
        acc, n_ok = np.zeros(len(blocks)), 0
        oof_pred = np.full(len(y), -1, dtype=int)
        for fold, (tr, te) in enumerate(grouped_splits(y, groups, N_SPLITS, seed)):
            rf = RandomForestClassifier(random_state=10_000 * seed + fold, **RF_PARAMS)
            rf.fit(Xs[wkey][tr], y[tr])
            oof_pred[te] = rf.predict(Xs[wkey][te])
            if len(np.unique(y[te])) < 2:
                warnings.warn(f"{wkey} seed={seed} fold={fold} 测试折单一类别,跳过")
                continue
            acc += block_permutation_importance(rf, Xs[wkey][te], y[te],
                                                blocks, BLOCK_REPEATS, rng)
            n_ok += 1
        rows.append(acc / max(n_ok, 1))
        oofs.append(balanced_accuracy_score(y, oof_pred))
        print(f"  {wkey:6s} seed={seed:<2d} OOF={oofs[-1]:.3f}  ({time.time()-t0:.0f}s)")
    per_seed_blk[wkey] = np.vstack(rows)
    oof_blk[wkey] = oofs
    print(f"  {wkey:6s} 平均 OOF = {np.mean(oofs):.3f} ± {np.std(oofs):.3f}\n")

np.savez(os.path.join(OUT_DIR, "block_perm.npz"),
         **{f"{w}__blockperm": per_seed_blk[w] for w in WINDOWS},
         **{f"{w}__oof": np.array(oof_blk[w]) for w in WINDOWS},
         block_members=np.array([",".join(map(str, b)) for b in blocks]),
         block_names=np.array([" + ".join(names[i] for i in b) for b in blocks]),
         block_kind=block_kind, block_size=sizes, seeds=np.array(SEEDS),
         block_distance=np.array([BLOCK_DISTANCE]))
print(f"总用时 {time.time()-t0:.0f}s -> {OUT_DIR}/block_perm.npz")

# 零值诊断 —— 这是判断掩盖问题有没有解决的直接证据
print("\n" + "=" * 60)
print("零值比例(逐特征 permutation vs block permutation)")
for w in WINDOWS:
    zf_feat = float((np.abs(per_seed_feat[w]["perm"]) < 1e-12).mean())
    zf_blk = float((np.abs(per_seed_blk[w]) < 1e-12).mean())
    arrow = "改善" if zf_blk < zf_feat - 0.05 else ("基本没变" if zf_blk < zf_feat + 0.05 else "变差")
    print(f"  {w:6s} 逐特征 {zf_feat:6.1%}  ->  分块 {zf_blk:6.1%}   {arrow}")
print("=" * 60)

  TS     seed=0  OOF=0.659  (381s)
  TS     seed=1  OOF=0.648  (761s)
  TS     seed=2  OOF=0.664  (1140s)
  TS     seed=3  OOF=0.666  (1520s)
  TS     seed=4  OOF=0.671  (1899s)
  TS     seed=5  OOF=0.663  (2279s)
  TS     seed=6  OOF=0.676  (2658s)
  TS     seed=7  OOF=0.683  (3038s)
  TS     seed=8  OOF=0.649  (3419s)
  TS     seed=9  OOF=0.656  (3799s)
  TS     平均 OOF = 0.664 ± 0.011

  late   seed=0  OOF=0.749  (4174s)
  late   seed=1  OOF=0.767  (4549s)
  late   seed=2  OOF=0.777  (4924s)
  late   seed=3  OOF=0.772  (5299s)
  late   seed=4  OOF=0.754  (5673s)
  late   seed=5  OOF=0.723  (6047s)
  late   seed=6  OOF=0.757  (6422s)
  late   seed=7  OOF=0.752  (6797s)
  late   seed=8  OOF=0.769  (7172s)
  late   seed=9  OOF=0.784  (7547s)
  late   平均 OOF = 0.761 ± 0.017

总用时 7547s -> /mnt/data1/student/trypsin/yucheng/07.12/v4/block_perm.npz

零值比例(逐特征 permutation vs block permutation)
  TS     逐特征  13.2%  ->  分块   6.0%   改善
  late   逐特征  42.5%  ->  分块  35.5%   改善


## Part 3 — 块级地板:成功了没有

和主流程同一套判据:把 10 个 seed 对半分成 5+5,各半平均,算同一窗口内两半之间的
top-15 Jaccard 和 Spearman ρ。

**关键看晚期窗口那一行。** 逐特征 permutation 在那里是 J = 0.034。

In [5]:
PARTS = [set(c) for c in combinations(range(len(SEEDS)), len(SEEDS) // 2)]
PARTS = PARTS[:len(PARTS) // 2]

def topk_jaccard(a, b, k=TOP_K):
    ta, tb = set(np.argsort(a)[::-1][:k]), set(np.argsort(b)[::-1][:k])
    return len(ta & tb) / len(ta | tb)

def split_half(M, k=TOP_K):
    rr, jj = [], []
    for A in PARTS:
        B = [i for i in range(len(M)) if i not in A]
        a, b = M[sorted(A)].mean(axis=0), M[B].mean(axis=0)
        rr.append(spearmanr(a, b).statistic); jj.append(topk_jaccard(a, b, k))
    return float(np.median(rr)), float(np.median(jj))

print(f"{'':22}{'地板 rho':>10}{'地板 Jaccard':>14}")
floors = {}
for w in WINDOWS:
    floors[("feat", w)] = split_half(per_seed_feat[w]["perm"])
    floors[("blk", w)] = split_half(per_seed_blk[w])
    print(f"  {w:6s} 逐特征 perm  {floors[('feat',w)][0]:>8.3f}{floors[('feat',w)][1]:>14.3f}")
    print(f"  {w:6s} block perm   {floors[('blk',w)][0]:>8.3f}{floors[('blk',w)][1]:>14.3f}")

print()
worst_feat = min(floors[("feat", w)][1] for w in WINDOWS)
worst_blk = min(floors[("blk", w)][1] for w in WINDOWS)
if worst_blk >= 0.30:
    VERDICT = ("成功:block permutation 在两个窗口都可复现。"
               "现在有了一个无偏且可用的方法,可以用它复核 SHAP 给出的 rho≈0.39,"
               "把上界变成真值。")
elif worst_blk > worst_feat + 0.10:
    VERDICT = (f"部分改善:最弱窗口地板从 {worst_feat:.3f} 升到 {worst_blk:.3f},"
               "但仍不够稳。可以调大 BLOCK_DISTANCE 让块更粗再试一次。")
else:
    VERDICT = (f"未改善(最弱窗口 {worst_feat:.3f} -> {worst_blk:.3f})。"
               "说明冗余不是唯一原因 —— 晚期窗口用无偏方法确实测不出来。"
               "报告里 SHAP 的『上界』说法就是最终结论,如实写明即可。")
print(">>>", VERDICT)

# 跨窗口:块级的重要特征集合变了多少
M = {w: per_seed_blk[w].mean(axis=0) for w in WINDOWS}
rho_x = spearmanr(M["TS"], M["late"]).statistic
jac_x = topk_jaccard(M["TS"], M["late"])
print(f"\n跨窗口(block perm): rho = {rho_x:+.3f}   top-{TOP_K} Jaccard = {jac_x:.3f}")
fa, fb = floors[("blk", "TS")][0], floors[("blk", "late")][0]
if min(fa, fb) >= 0.20:
    print(f"  衰减校正后 rho = {rho_x / np.sqrt(fa * fb):+.3f}"
          f"   (对比 SHAP 的 +0.39 / Gini 的 +0.41)")
else:
    print(f"  地板 {fa:.3f}/{fb:.3f} 低于 0.20,不做衰减校正(近零数相除不可信)")

                          地板 rho    地板 Jaccard
  TS     逐特征 perm     0.116         0.200
  TS     block perm      0.305         0.304
  late   逐特征 perm     0.128         0.034
  late   block perm      0.275         0.200

>>> 部分改善:最弱窗口地板从 0.034 升到 0.200,但仍不够稳。可以调大 BLOCK_DISTANCE 让块更粗再试一次。

跨窗口(block perm): rho = +0.172   top-15 Jaccard = 0.250
  衰减校正后 rho = +0.594   (对比 SHAP 的 +0.39 / Gini 的 +0.41)


## Part 4 — 块级的水富集 + top 块

报告里最强的结论是"水在 TS 占 top-50 的 26%,late 只剩 4%"。这里用块级重新检验一次:
如果 block permutation 也给出同样的方向,这条结论就同时被**有偏(Gini/SHAP)和无偏
(block perm)**两类方法支持,是最硬的证据。

块的数量少于 274,所以 top-N 相应缩小(取块数的前 20%)。

In [6]:
TOP_N = max(10, int(0.20 * len(blocks)))
n_water_blocks = int((block_kind == "water").sum())
base_rate = n_water_blocks / len(blocks)
print(f"块级水富集(top-{TOP_N} / {len(blocks)} 块,基线 {base_rate:.1%})\n")

cnt = {}
for w in WINDOWS:
    top = np.argsort(M[w])[::-1][:TOP_N]
    cnt[w] = int((block_kind[top] == "water").sum())
odds, p = fisher_exact([[cnt["TS"], TOP_N - cnt["TS"]],
                        [cnt["late"], TOP_N - cnt["late"]]])
print(f"  block perm : TS {cnt['TS']:>2}/{TOP_N}  vs  late {cnt['late']:>2}/{TOP_N}"
      f"   OR={odds:.2f}  Fisher p={p:.2e}")
print("  参照(逐特征 top-50):Gini 11 vs 1 (p=0.004) | SHAP 13 vs 2 (p=0.004) | "
      "perm 14 vs 7 (p=0.14)")
if p < 0.05 and odds > 1:
    print("\n  >>> 无偏方法也支持水的 regime 转移 —— 这条结论现在被两类方法共同支持,"
          "是报告里最硬的一条。")
elif odds > 1:
    print(f"\n  >>> 方向一致但未达显著(p={p:.3f})。块数少导致检验功效下降,"
          "可以调小 BLOCK_DISTANCE 增加块数再试。")
else:
    print("\n  >>> 无偏方法不支持。报告里这条结论要加限定:"
          "仅在与 Gini 同源的方法下成立。")

for w in WINDOWS:
    a, b = WINDOWS[w]
    print(f"\n[{w}] frames {a}-{b-1}  top-8 块 (block permutation):")
    for r, i in enumerate(np.argsort(M[w])[::-1][:8], 1):
        nm = " + ".join(names[j] for j in blocks[i][:4])
        if len(blocks[i]) > 4:
            nm += f" ... (+{len(blocks[i])-4})"
        print(f"   {r}. [{block_kind[i]:7s} size={sizes[i]:2d}] {nm}   {M[w][i]:.5f}")

块级水富集(top-14 / 73 块,基线 67.1%)

  block perm : TS 11/14  vs  late  8/14   OR=2.75  Fisher p=4.20e-01
  参照(逐特征 top-50):Gini 11 vs 1 (p=0.004) | SHAP 13 vs 2 (p=0.004) | perm 14 vs 7 (p=0.14)

  >>> 方向一致但未达显著(p=0.420)。块数少导致检验功效下降,可以调小 BLOCK_DISTANCE 增加块数再试。

[TS] frames 500-1500  top-8 块 (block permutation):
   1. [protein size=37] ALA56 + ALA57 + HIS58 + CYS59 ... (+33)   0.06702
   2. [protein size=74] VAL20 + GLY21 + GLY22 + SER122 ... (+70)   0.02515
   3. [protein size=18] LEU36 + ASN37 + SER38 + GLY39 ... (+14)   0.01976
   4. [water   size= 1] water rank28   0.00880
   5. [water   size= 1] water rank41   0.00571
   6. [water   size= 1] water rank2   0.00278
   7. [water   size= 1] water rank3   0.00251
   8. [water   size= 1] water rank43   0.00210

[late] frames 2000-2499  top-8 块 (block permutation):
   1. [protein size=74] VAL20 + GLY21 + GLY22 + SER122 ... (+70)   0.08988
   2. [protein size=37] ALA56 + ALA57 + HIS58 + CYS59 ... (+33)   0.06422
   3. [protein size=18] LEU36 + A

## Part 5 — 汇总表 + 落盘

一张可以直接贴进报告的对照表。

In [7]:
import csv
summary = {
    "block_distance": BLOCK_DISTANCE, "separate_waters": SEPARATE_WATERS,
    "n_blocks": len(blocks), "block_repeats": BLOCK_REPEATS,
    "floors": {f"{k[0]}|{k[1]}": {"rho": v[0], "jaccard": v[1]} for k, v in floors.items()},
    "cross_window_blockperm": {"rho": float(rho_x), "jaccard": float(jac_x)},
    "oof": {w: float(np.mean(oof_blk[w])) for w in WINDOWS},
    "water_enrichment_blocklevel": {"top_n": TOP_N, "TS": cnt["TS"], "late": cnt["late"],
                                    "odds_ratio": float(odds), "fisher_p": float(p)},
    "verdict": VERDICT,
}
json.dump(summary, open(os.path.join(OUT_DIR, "block_perm_summary.json"), "w"),
          indent=2, ensure_ascii=False)

with open(os.path.join(OUT_DIR, "block_perm_table.csv"), "w", newline="", encoding="utf-8") as fh:
    wtr = csv.writer(fh)
    wtr.writerow(["block_id", "size", "kind", "members"] + [f"blockperm_{w}" for w in WINDOWS])
    for i, b in enumerate(blocks):
        wtr.writerow([i, sizes[i], block_kind[i], " + ".join(names[j] for j in b)]
                     + [f"{M[w][i]:.8g}" for w in WINDOWS])

print("=" * 74)
print(f"{'方法':<26}{'TS 地板 J':>12}{'late 地板 J':>13}{'跨窗口 J':>11}")
print("-" * 74)
print(f"{'逐特征 permutation':<26}{floors[('feat','TS')][1]:>12.3f}"
      f"{floors[('feat','late')][1]:>13.3f}{'—':>11}")
print(f"{'block permutation':<26}{floors[('blk','TS')][1]:>12.3f}"
      f"{floors[('blk','late')][1]:>13.3f}{jac_x:>11.3f}")
print(f"{'SHAP (有偏,同源 Gini)':<26}{0.875:>12.3f}{1.000:>13.3f}{0.200:>11.3f}")
print("=" * 74)
print(VERDICT)
print(f"\n-> {OUT_DIR}/block_perm_summary.json")
print(f"-> {OUT_DIR}/block_perm_table.csv")
print(f"-> {OUT_DIR}/block_perm.npz")

方法                             TS 地板 J    late 地板 J      跨窗口 J
--------------------------------------------------------------------------
逐特征 permutation                  0.200        0.034          —
block permutation                0.304        0.200      0.250
SHAP (有偏,同源 Gini)                0.875        1.000      0.200
部分改善:最弱窗口地板从 0.034 升到 0.200,但仍不够稳。可以调大 BLOCK_DISTANCE 让块更粗再试一次。

-> /mnt/data1/student/trypsin/yucheng/07.12/v4/block_perm_summary.json
-> /mnt/data1/student/trypsin/yucheng/07.12/v4/block_perm_table.csv
-> /mnt/data1/student/trypsin/yucheng/07.12/v4/block_perm.npz
